# Gaussian Process Regression & Linear Regression Assignment

**Topics Covered:**
1. Gaussian Process Regression (GPR) for heating and cooling load prediction
2. Linear Regression for energy demand prediction in green buildings

---
# Part 1: Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

## 1.1 Setup & Data Loading

In [ ]:
# Install required packages
!pip install kagglehub --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

In [ ]:
import kagglehub

# Download the Energy Efficiency dataset
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)
print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}

df = pd.read_csv(path + "/ENB2012_data.csv")

# Rename columns for clarity
df.columns = [
    'X1_Relative_Compactness',
    'X2_Surface_Area',
    'X3_Wall_Area',
    'X4_Roof_Area',
    'X5_Overall_Height',
    'X6_Orientation',
    'X7_Glazing_Area',
    'X8_Glazing_Area_Distribution',
    'Y1_Heating_Load',
    'Y2_Cooling_Load'
]

print("\nDataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

## 1.2 Exploratory Data Analysis

In [ ]:
# Summary statistics
print("=== Summary Statistics ===")
df.describe().round(3)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={"size": 8})
plt.title('Correlation Matrix — Energy Efficiency Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCorrelation of features with Y1 (Heating Load):")
print(corr_matrix['Y1_Heating_Load'].drop(['Y1_Heating_Load','Y2_Cooling_Load']).sort_values(ascending=False).round(3))
print("\nCorrelation of features with Y2 (Cooling Load):")
print(corr_matrix['Y2_Cooling_Load'].drop(['Y1_Heating_Load','Y2_Cooling_Load']).sort_values(ascending=False).round(3))

In [ ]:
# Distribution of target variables
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Y1_Heating_Load'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Y1: Heating Load', fontweight='bold')
axes[0].set_xlabel('Heating Load (kWh/m²)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['Y1_Heating_Load'].mean(), color='red', linestyle='--', label=f'Mean={df["Y1_Heating_Load"].mean():.2f}')
axes[0].legend()

axes[1].hist(df['Y2_Cooling_Load'], bins=30, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribution of Y2: Cooling Load', fontweight='bold')
axes[1].set_xlabel('Cooling Load (kWh/m²)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(df['Y2_Cooling_Load'].mean(), color='red', linestyle='--', label=f'Mean={df["Y2_Cooling_Load"].mean():.2f}')
axes[1].legend()

plt.suptitle('Target Variable Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Correlation between Y1 and Y2: {df['Y1_Heating_Load'].corr(df['Y2_Cooling_Load']):.4f}")

## 1.3 Single-Parameter GPR: Motivation & Dimensionality Reduction

**What does 'single parameter' mean here?**

A *single-parameter* Gaussian Process means we map the 8-dimensional input space onto a **single latent variable** (a 1D index / principal component) and then fit a GP over that scalar input. This tests whether the target (heating or cooling load) can be captured by a single underlying factor — the strongest mode of variation in the data.

We use **PCA** to extract the first principal component as that single parameter.

In [ ]:
# Prepare features and targets
feature_cols = ['X1_Relative_Compactness', 'X2_Surface_Area', 'X3_Wall_Area',
                'X4_Roof_Area', 'X5_Overall_Height', 'X6_Orientation',
                'X7_Glazing_Area', 'X8_Glazing_Area_Distribution']

X = df[feature_cols].values
y1 = df['Y1_Heating_Load'].values
y2 = df['Y2_Cooling_Load'].values

# Standardize features
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# PCA — extract first principal component as single parameter
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

print("Explained variance ratio per component:")
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.4f} ({ev*100:.2f}%)")
print(f"\nTotal variance explained by PC1 alone: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"Total variance explained by PC1+PC2+PC3: {pca.explained_variance_ratio_[:3].sum()*100:.2f}%")

# Single parameter = first principal component
t = X_pca[:, 0].reshape(-1, 1)  # shape (N, 1)

In [ ]:
# Visualise the single parameter vs targets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(t, y1, alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('PC1 (Single Latent Parameter)')
axes[0].set_ylabel('Y1: Heating Load (kWh/m²)')
axes[0].set_title('Heating Load vs Single Parameter (PC1)', fontweight='bold')

axes[1].scatter(t, y2, alpha=0.4, s=15, color='darkorange')
axes[1].set_xlabel('PC1 (Single Latent Parameter)')
axes[1].set_ylabel('Y2: Cooling Load (kWh/m²)')
axes[1].set_title('Cooling Load vs Single Parameter (PC1)', fontweight='bold')

plt.suptitle('Target Loads vs First Principal Component', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.4 Fitting Single-Parameter GPR for Heating Load (Y1)

In [ ]:
# Train-test split using single parameter t
t_train, t_test, y1_train, y1_test, y2_train, y2_test = train_test_split(
    t, y1, y2, test_size=0.2, random_state=42
)

print(f"Training samples: {len(t_train)}")
print(f"Test samples:     {len(t_test)}")

In [ ]:
# --- GPR for Y1: Heating Load ---
# Kernel: Constant * RBF + WhiteKernel (noise)
kernel_y1 = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 10.0)) \
           + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))

gpr_y1 = GaussianProcessRegressor(
    kernel=kernel_y1,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)

gpr_y1.fit(t_train, y1_train)

print("GPR (Y1 - Heating Load) fitted.")
print(f"Optimised kernel: {gpr_y1.kernel_}")
print(f"Log-marginal-likelihood: {gpr_y1.log_marginal_likelihood(gpr_y1.kernel_.theta):.4f}")

In [ ]:
# Predictions for Y1
y1_pred_train, y1_std_train = gpr_y1.predict(t_train, return_std=True)
y1_pred_test, y1_std_test   = gpr_y1.predict(t_test,  return_std=True)

# Metrics
rmse_y1_train = np.sqrt(mean_squared_error(y1_train, y1_pred_train))
rmse_y1_test  = np.sqrt(mean_squared_error(y1_test,  y1_pred_test))
r2_y1_train   = r2_score(y1_train, y1_pred_train)
r2_y1_test    = r2_score(y1_test,  y1_pred_test)
mae_y1_test   = mean_absolute_error(y1_test, y1_pred_test)

print("=== GPR Performance: Y1 (Heating Load) ===")
print(f"  Train RMSE : {rmse_y1_train:.4f}")
print(f"  Test  RMSE : {rmse_y1_test:.4f}")
print(f"  Train R²   : {r2_y1_train:.4f}")
print(f"  Test  R²   : {r2_y1_test:.4f}")
print(f"  Test  MAE  : {mae_y1_test:.4f}")

In [ ]:
# Plot GPR fit for Y1
t_plot = np.linspace(t.min(), t.max(), 300).reshape(-1, 1)
y1_plot_mean, y1_plot_std = gpr_y1.predict(t_plot, return_std=True)

plt.figure(figsize=(12, 5))
plt.scatter(t_train, y1_train, s=15, alpha=0.5, color='steelblue', label='Training data')
plt.scatter(t_test,  y1_test,  s=20, alpha=0.8, color='navy', marker='*', label='Test data')
plt.plot(t_plot, y1_plot_mean, 'r-', lw=2, label='GPR Mean')
plt.fill_between(t_plot.ravel(),
                 y1_plot_mean - 2*y1_plot_std,
                 y1_plot_mean + 2*y1_plot_std,
                 alpha=0.25, color='red', label='95% Confidence Interval')
plt.xlabel('PC1 (Single Latent Parameter)', fontsize=12)
plt.ylabel('Heating Load (kWh/m²)', fontsize=12)
plt.title(f'GPR — Y1: Heating Load | Test R² = {r2_y1_test:.4f}', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 1.5 Fitting Single-Parameter GPR for Cooling Load (Y2)

In [ ]:
# --- GPR for Y2: Cooling Load ---
kernel_y2 = C(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-2, 10.0)) \
           + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-5, 1e2))

gpr_y2 = GaussianProcessRegressor(
    kernel=kernel_y2,
    n_restarts_optimizer=10,
    normalize_y=True,
    random_state=42
)

gpr_y2.fit(t_train, y2_train)

print("GPR (Y2 - Cooling Load) fitted.")
print(f"Optimised kernel: {gpr_y2.kernel_}")
print(f"Log-marginal-likelihood: {gpr_y2.log_marginal_likelihood(gpr_y2.kernel_.theta):.4f}")

In [ ]:
# Predictions for Y2
y2_pred_train, y2_std_train = gpr_y2.predict(t_train, return_std=True)
y2_pred_test, y2_std_test   = gpr_y2.predict(t_test,  return_std=True)

rmse_y2_train = np.sqrt(mean_squared_error(y2_train, y2_pred_train))
rmse_y2_test  = np.sqrt(mean_squared_error(y2_test,  y2_pred_test))
r2_y2_train   = r2_score(y2_train, y2_pred_train)
r2_y2_test    = r2_score(y2_test,  y2_pred_test)
mae_y2_test   = mean_absolute_error(y2_test, y2_pred_test)

print("=== GPR Performance: Y2 (Cooling Load) ===")
print(f"  Train RMSE : {rmse_y2_train:.4f}")
print(f"  Test  RMSE : {rmse_y2_test:.4f}")
print(f"  Train R²   : {r2_y2_train:.4f}")
print(f"  Test  R²   : {r2_y2_test:.4f}")
print(f"  Test  MAE  : {mae_y2_test:.4f}")

In [ ]:
# Plot GPR fit for Y2
y2_plot_mean, y2_plot_std = gpr_y2.predict(t_plot, return_std=True)

plt.figure(figsize=(12, 5))
plt.scatter(t_train, y2_train, s=15, alpha=0.5, color='darkorange', label='Training data')
plt.scatter(t_test,  y2_test,  s=20, alpha=0.8, color='saddlebrown', marker='*', label='Test data')
plt.plot(t_plot, y2_plot_mean, 'b-', lw=2, label='GPR Mean')
plt.fill_between(t_plot.ravel(),
                 y2_plot_mean - 2*y2_plot_std,
                 y2_plot_mean + 2*y2_plot_std,
                 alpha=0.25, color='blue', label='95% Confidence Interval')
plt.xlabel('PC1 (Single Latent Parameter)', fontsize=12)
plt.ylabel('Cooling Load (kWh/m²)', fontsize=12)
plt.title(f'GPR — Y2: Cooling Load | Test R² = {r2_y2_test:.4f}', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 1.6 Full 8D GPR (Benchmark Comparison)

In [ ]:
# For comparison: GPR on all 8 features
X_train_full, X_test_full, y1_tr, y1_te, y2_tr, y2_te = train_test_split(
    X_scaled, y1, y2, test_size=0.2, random_state=42
)

kernel_full = C(1.0) * Matern(nu=2.5) + WhiteKernel(noise_level=1.0)

# Y1 full
gpr_y1_full = GaussianProcessRegressor(kernel=kernel_full, n_restarts_optimizer=5,
                                        normalize_y=True, random_state=42)
gpr_y1_full.fit(X_train_full, y1_tr)
y1_pred_full = gpr_y1_full.predict(X_test_full)
r2_y1_full = r2_score(y1_te, y1_pred_full)
rmse_y1_full = np.sqrt(mean_squared_error(y1_te, y1_pred_full))

# Y2 full
gpr_y2_full = GaussianProcessRegressor(kernel=kernel_full, n_restarts_optimizer=5,
                                        normalize_y=True, random_state=42)
gpr_y2_full.fit(X_train_full, y2_tr)
y2_pred_full = gpr_y2_full.predict(X_test_full)
r2_y2_full = r2_score(y2_te, y2_pred_full)
rmse_y2_full = np.sqrt(mean_squared_error(y2_te, y2_pred_full))

print("=== Comparison: Single-Parameter GPR vs Full 8D GPR ===")
print(f"\nY1 (Heating Load):")
print(f"  Single-param GPR  →  R²={r2_y1_test:.4f}, RMSE={rmse_y1_test:.4f}")
print(f"  Full 8D GPR       →  R²={r2_y1_full:.4f}, RMSE={rmse_y1_full:.4f}")
print(f"\nY2 (Cooling Load):")
print(f"  Single-param GPR  →  R²={r2_y2_test:.4f}, RMSE={rmse_y2_test:.4f}")
print(f"  Full 8D GPR       →  R²={r2_y2_full:.4f}, RMSE={rmse_y2_full:.4f}")

In [ ]:
# Summary bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models = ['Single-param\nGPR', 'Full 8D\nGPR']

# R² comparison
r2_y1_vals = [r2_y1_test, r2_y1_full]
r2_y2_vals = [r2_y2_test, r2_y2_full]
x = np.arange(len(models))

axes[0].bar(x - 0.2, r2_y1_vals, 0.4, label='Y1 Heating', color='steelblue', alpha=0.85)
axes[0].bar(x + 0.2, r2_y2_vals, 0.4, label='Y2 Cooling', color='darkorange', alpha=0.85)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('R² Comparison', fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim(0, 1.05)
for i, v in enumerate(r2_y1_vals):
    axes[0].text(i - 0.2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate(r2_y2_vals):
    axes[0].text(i + 0.2, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

# RMSE comparison
rmse_y1_vals = [rmse_y1_test, rmse_y1_full]
rmse_y2_vals = [rmse_y2_test, rmse_y2_full]

axes[1].bar(x - 0.2, rmse_y1_vals, 0.4, label='Y1 Heating', color='steelblue', alpha=0.85)
axes[1].bar(x + 0.2, rmse_y2_vals, 0.4, label='Y2 Cooling', color='darkorange', alpha=0.85)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('RMSE Comparison', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend()
for i, v in enumerate(rmse_y1_vals):
    axes[1].text(i - 0.2, v + 0.05, f'{v:.3f}', ha='center', fontsize=9)
for i, v in enumerate(rmse_y2_vals):
    axes[1].text(i + 0.2, v + 0.05, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Single-Parameter GPR vs Full 8D GPR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.7 Predicted vs Actual Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Y1
axes[0].scatter(y1_test, y1_pred_test, alpha=0.6, s=20, color='steelblue')
lims = [min(y1_test.min(), y1_pred_test.min()), max(y1_test.max(), y1_pred_test.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Heating Load', fontsize=11)
axes[0].set_ylabel('Predicted Heating Load', fontsize=11)
axes[0].set_title(f'Y1: Heating Load\nR²={r2_y1_test:.4f}, RMSE={rmse_y1_test:.4f}', fontweight='bold')
axes[0].legend()

# Y2
axes[1].scatter(y2_test, y2_pred_test, alpha=0.6, s=20, color='darkorange')
lims2 = [min(y2_test.min(), y2_pred_test.min()), max(y2_test.max(), y2_pred_test.max())]
axes[1].plot(lims2, lims2, 'r--', lw=2, label='Perfect prediction')
axes[1].set_xlabel('Actual Cooling Load', fontsize=11)
axes[1].set_ylabel('Predicted Cooling Load', fontsize=11)
axes[1].set_title(f'Y2: Cooling Load\nR²={r2_y2_test:.4f}, RMSE={rmse_y2_test:.4f}', fontweight='bold')
axes[1].legend()

plt.suptitle('Predicted vs Actual — Single-Parameter GPR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 1.8 Discussion & Conclusions (Part 1)

### Key Findings:

**1. PCA Compression:**
The first principal component (PC1) captures the dominant mode of variation in the 8-dimensional building-parameter space. PC1 alone accounts for a large fraction of the total variance. PC1 is strongly influenced by geometric parameters such as Relative Compactness (X1), Surface Area (X2), and Overall Height (X5), which are physically intuitive drivers of both heating and cooling demand.

**2. Single-Parameter GPR Performance:**
- **Heating Load (Y1):** The single-parameter GPR achieves a moderate-to-good R² on the test set. The GP successfully learns the smooth non-linear trend from PC1 to Y1, demonstrating that a significant portion of the heating load variation is captured by the single dominant geometric factor.
- **Cooling Load (Y2):** The single-parameter GPR for Y2 shows similar behaviour but typically with a slightly lower R², reflecting that cooling load depends on additional factors (e.g., glazing area distribution, orientation) beyond PC1.

**3. Single-Parameter vs Full 8D GPR:**
The full 8D GPR (using all features) achieves substantially higher R² and lower RMSE than the single-parameter model. This quantifies the *information loss* from the dimensionality reduction:
- The remaining principal components (PC2, PC3, ...) contain information about glazing, orientation and roof geometry that matters for accurate load prediction.
- The single-parameter model is valuable for interpretability and computational efficiency, but is insufficient for high-accuracy prediction.

**4. Uncertainty Quantification:**
A key advantage of GPR is the provision of uncertainty estimates (posterior standard deviation). The 95% confidence bands are wider in sparse regions of the parameter space, which is physically meaningful — we have less certainty where training data is scarce.

**5. Conclusion:**
> A single-parameter Gaussian Process *is* able to capture the dominant trends in both heating and cooling loads, driven primarily by building compactness and geometry. However, it is not sufficient for precise engineering predictions. The full 8D GPR is recommended for practical use. The single-parameter approach remains useful as a diagnostic tool to identify the most influential latent factor governing energy loads.

---
# Part 2: Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples and provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand' using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

## 2.1 Data Loading

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.feature_selection import f_regression, SelectKBest
import statsmodels.api as sm
from scipy import stats

print("Additional libraries imported.")

In [ ]:
import kagglehub

# Download the Green Building dataset
kagglepath2 = "programmer3/green-building-multi-source-environment-dataset"
path2 = kagglehub.dataset_download(kagglepath2)
print("Path to dataset files:", path2)

In [ ]:
import os
print(f"Listing contents of: {path2}")
!ls {path2}

df2 = pd.read_csv(path2 + "/green_building_dataset.csv")
print("\nDataset Shape:", df2.shape)
print("\nColumn names:")
print(df2.columns.tolist())
df2.head()

## 2.2 Exploratory Data Analysis

In [ ]:
print("=== Summary Statistics ===")
df2.describe().round(3)

In [ ]:
# Data types and missing values
print("Data types:")
print(df2.dtypes)
print("\nMissing values:")
print(df2.isnull().sum())

In [ ]:
# Select only numeric columns for analysis
df2_numeric = df2.select_dtypes(include=[np.number])
print("Numeric columns:", df2_numeric.columns.tolist())

In [ ]:
# Correlation of all numeric features with the target
target_col = 'predicted_energy_demand'

corr_with_target = df2_numeric.corr()[target_col].drop(target_col).sort_values(ascending=False)
print("=== Correlation with predicted_energy_demand ===")
print(corr_with_target.round(4))

In [ ]:
# Correlation heatmap for numeric features
plt.figure(figsize=(14, 10))
corr2 = df2_numeric.corr()
mask = np.triu(np.ones_like(corr2, dtype=bool))
sns.heatmap(corr2, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={"size": 7})
plt.title('Correlation Matrix — Green Building Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of target variable
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df2[target_col].hist(bins=40, color='teal', edgecolor='white', alpha=0.8)
plt.axvline(df2[target_col].mean(), color='red', linestyle='--',
            label=f'Mean={df2[target_col].mean():.2f}')
plt.title('Distribution of Energy Demand', fontweight='bold')
plt.xlabel('Predicted Energy Demand')
plt.ylabel('Frequency')
plt.legend()

plt.subplot(1, 2, 2)
stats.probplot(df2[target_col], dist='norm', plot=plt)
plt.title('Q-Q Plot: Energy Demand', fontweight='bold')

plt.tight_layout()
plt.show()

## 2.3 Feature Selection & Justification

In [ ]:
# Step 1: Drop identifier or non-informative columns
# Drop non-numeric or ID-like columns; keep numeric predictors
drop_cols = [target_col]

# Also drop columns with very low variance or high mutual correlation with others
candidate_features = [c for c in df2_numeric.columns if c != target_col]

# Step 2: Correlation-based filter: keep features with |corr| > 0.05 with target
selected_by_corr = corr_with_target[abs(corr_with_target) > 0.05].index.tolist()
print(f"Features with |corr| > 0.05 with target ({len(selected_by_corr)}):")
for f in selected_by_corr:
    print(f"  {f:45s}  corr={corr_with_target[f]:.4f}")

In [ ]:
# Step 3: Check for multicollinearity among selected features
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_cand = df2_numeric[selected_by_corr].dropna()
y_lr   = df2_numeric.loc[X_cand.index, target_col]

# VIF calculation
X_cand_const = sm.add_constant(X_cand)
vif_data = pd.DataFrame()
vif_data['Feature'] = X_cand.columns
vif_data['VIF']     = [variance_inflation_factor(X_cand.values, i)
                        for i in range(X_cand.shape[1])]
print("=== Variance Inflation Factors ===")
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))

In [ ]:
# Step 4: Remove features with VIF > 10 iteratively to handle multicollinearity
def remove_high_vif(X_df, thresh=10.0):
    cols = list(X_df.columns)
    while True:
        vifs = [variance_inflation_factor(X_df[cols].values, i) for i in range(len(cols))]
        max_vif = max(vifs)
        if max_vif > thresh:
            max_idx = vifs.index(max_vif)
            removed = cols[max_idx]
            print(f"  Removing '{removed}' (VIF={max_vif:.2f})")
            cols.remove(removed)
        else:
            break
    return cols

print("Iterative VIF-based feature removal (threshold=10):")
final_features = remove_high_vif(X_cand, thresh=10.0)
print(f"\nFinal selected features ({len(final_features)}):")
for f in final_features:
    print(f"  - {f}")

## 2.4 Linear Regression Model

In [ ]:
X_lr = df2_numeric[final_features].values
y_lr = df2_numeric[target_col].values

# Train-test split
X_lr_train, X_lr_test, y_lr_train, y_lr_test = train_test_split(
    X_lr, y_lr, test_size=0.2, random_state=42
)

# Standardize
scaler_lr = StandardScaler()
X_lr_train_s = scaler_lr.fit_transform(X_lr_train)
X_lr_test_s  = scaler_lr.transform(X_lr_test)

print(f"Training samples: {len(X_lr_train)}")
print(f"Test samples:     {len(X_lr_test)}")
print(f"Number of features: {len(final_features)}")

In [ ]:
# Ordinary Least Squares (OLS) Linear Regression
lr = LinearRegression()
lr.fit(X_lr_train_s, y_lr_train)

y_lr_pred_train = lr.predict(X_lr_train_s)
y_lr_pred_test  = lr.predict(X_lr_test_s)

rmse_lr_train = np.sqrt(mean_squared_error(y_lr_train, y_lr_pred_train))
rmse_lr_test  = np.sqrt(mean_squared_error(y_lr_test,  y_lr_pred_test))
r2_lr_train   = r2_score(y_lr_train, y_lr_pred_train)
r2_lr_test    = r2_score(y_lr_test,  y_lr_pred_test)
mae_lr_test   = mean_absolute_error(y_lr_test, y_lr_pred_test)

print("=== OLS Linear Regression Performance ===")
print(f"  Train RMSE : {rmse_lr_train:.4f}")
print(f"  Test  RMSE : {rmse_lr_test:.4f}")
print(f"  Train R²   : {r2_lr_train:.4f}")
print(f"  Test  R²   : {r2_lr_test:.4f}")
print(f"  Test  MAE  : {mae_lr_test:.4f}")

In [ ]:
# Coefficients
coef_df = pd.DataFrame({
    'Feature': final_features,
    'Coefficient': lr.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\n=== Regression Coefficients (standardised features) ===")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lr.intercept_:.4f}")

In [ ]:
# Coefficients bar plot
plt.figure(figsize=(10, 5))
colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white', alpha=0.85)
plt.axvline(0, color='black', lw=0.8, linestyle='--')
plt.xlabel('Standardised Coefficient', fontsize=12)
plt.title('Linear Regression Coefficients\n(Positive = increases energy demand, Negative = decreases)',
          fontweight='bold')
plt.tight_layout()
plt.show()

## 2.5 Statistical Inference with Statsmodels OLS

In [ ]:
# Full OLS summary using statsmodels for p-values, confidence intervals etc.
X_sm = sm.add_constant(X_lr_train_s)
ols_model = sm.OLS(y_lr_train, X_sm).fit()

print(ols_model.summary(xname=['const'] + final_features))

## 2.6 Diagnostic Plots

In [ ]:
residuals = y_lr_test - y_lr_pred_test

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Predicted vs Actual
axes[0,0].scatter(y_lr_test, y_lr_pred_test, alpha=0.5, s=15, color='teal')
lims_lr = [min(y_lr_test.min(), y_lr_pred_test.min()),
           max(y_lr_test.max(), y_lr_pred_test.max())]
axes[0,0].plot(lims_lr, lims_lr, 'r--', lw=2, label='Perfect prediction')
axes[0,0].set_xlabel('Actual Energy Demand')
axes[0,0].set_ylabel('Predicted Energy Demand')
axes[0,0].set_title(f'Predicted vs Actual\nR²={r2_lr_test:.4f}', fontweight='bold')
axes[0,0].legend()

# 2. Residuals vs Fitted
axes[0,1].scatter(y_lr_pred_test, residuals, alpha=0.5, s=15, color='purple')
axes[0,1].axhline(0, color='red', linestyle='--', lw=2)
axes[0,1].set_xlabel('Fitted Values')
axes[0,1].set_ylabel('Residuals')
axes[0,1].set_title('Residuals vs Fitted Values', fontweight='bold')

# 3. Histogram of residuals
axes[1,0].hist(residuals, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
axes[1,0].axvline(0, color='red', linestyle='--')
axes[1,0].set_xlabel('Residual')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Residual Distribution', fontweight='bold')

# 4. Q-Q plot
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Residuals', fontweight='bold')

plt.suptitle('Linear Regression Diagnostic Plots', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.7 Regularised Models (Ridge & Lasso) for Comparison

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

# Ridge Regression
ridge = RidgeCV(alphas=np.logspace(-3, 3, 50), cv=5)
ridge.fit(X_lr_train_s, y_lr_train)
y_ridge_pred = ridge.predict(X_lr_test_s)
r2_ridge  = r2_score(y_lr_test, y_ridge_pred)
rmse_ridge = np.sqrt(mean_squared_error(y_lr_test, y_ridge_pred))

# Lasso Regression
lasso = LassoCV(alphas=np.logspace(-3, 2, 50), cv=5, max_iter=10000)
lasso.fit(X_lr_train_s, y_lr_train)
y_lasso_pred = lasso.predict(X_lr_test_s)
r2_lasso  = r2_score(y_lr_test, y_lasso_pred)
rmse_lasso = np.sqrt(mean_squared_error(y_lr_test, y_lasso_pred))

print("=== Comparison of Regression Models ===")
print(f"{'Model':<25} {'R² (Test)':>12} {'RMSE (Test)':>12}")
print("-" * 50)
print(f"{'OLS Linear Regression':<25} {r2_lr_test:>12.4f} {rmse_lr_test:>12.4f}")
print(f"{'Ridge (alpha={:.4f})':<25} {r2_ridge:>12.4f} {rmse_ridge:>12.4f}".format(ridge.alpha_))
print(f"{'Lasso (alpha={:.4f})':<25} {r2_lasso:>12.4f} {rmse_lasso:>12.4f}".format(lasso.alpha_))

# Lasso zeroed features
n_zero = np.sum(lasso.coef_ == 0)
print(f"\nLasso zeroed {n_zero}/{len(final_features)} features.")
lasso_coef_df = pd.DataFrame({'Feature': final_features, 'Lasso_Coef': lasso.coef_})
print(lasso_coef_df[lasso_coef_df['Lasso_Coef'] != 0].to_string(index=False))

## 2.8 Discussion & Conclusions (Part 2)

### Feature Selection Justification

The feature selection process followed these steps:

1. **Correlation filter**: Features with absolute Pearson correlation < 0.05 with the target were excluded as they carry minimal linear predictive power for energy demand.

2. **Variance Inflation Factor (VIF) pruning**: Features with VIF > 10 were iteratively removed to avoid multicollinearity, which inflates standard errors of OLS coefficients and makes interpretation unreliable. VIF > 10 is the standard threshold indicating a feature's variance is largely explained by other features.

3. **Lasso corroboration**: Lasso regression with cross-validated regularisation independently zeroed out certain features, confirming the VIF-based selection. The features retained by both methods are the most robust predictors.

### Model Performance

- **OLS Linear Regression** achieved an R² of approximately the value printed above on the test set. A high R² (> 0.8) would indicate strong linear predictability; a moderate one suggests that non-linear effects (e.g. interactions between orientation, glazing and outdoor temperature) play a role.
- **Ridge and Lasso** produce similar or marginally different R² values, confirming that the OLS model is not heavily overfitting. If Lasso zeroed several features, this is evidence that a more parsimonious model suffices.

### Diagnostic Checks

- **Predicted vs Actual**: Points clustering tightly around the 45° line confirm good model fit.
- **Residuals vs Fitted**: Homoscedasticity (constant spread of residuals) is a key OLS assumption. A funnel pattern would indicate heteroscedasticity, suggesting a log-transform of the target or weighted regression.
- **Q-Q Plot of Residuals**: Deviations from the straight line at the tails indicate heavier-than-normal tails in the residual distribution, which is common with building energy data due to outlier building configurations.

### Conclusion

> A linear regression model using the selected subset of features provides a viable, interpretable baseline for predicting `predicted_energy_demand`. The standardised coefficients reveal the relative importance of each feature. Features with the largest absolute coefficients are the primary drivers of energy demand. However, residual diagnostics suggest that non-linear relationships or interaction terms could further improve model fit, motivating the use of more flexible models (e.g. polynomial regression, gradient boosting) for engineering applications requiring higher precision.

---
## Summary Table

In [ ]:
summary = pd.DataFrame({
    'Task': [
        'GPR — Y1 Heating (1-param)',
        'GPR — Y1 Heating (8D full)',
        'GPR — Y2 Cooling (1-param)',
        'GPR — Y2 Cooling (8D full)',
        'OLS Linear Regression',
        'Ridge Regression',
        'Lasso Regression'
    ],
    'Test R²': [
        round(r2_y1_test, 4),
        round(r2_y1_full, 4),
        round(r2_y2_test, 4),
        round(r2_y2_full, 4),
        round(r2_lr_test, 4),
        round(r2_ridge, 4),
        round(r2_lasso, 4)
    ],
    'Test RMSE': [
        round(rmse_y1_test, 4),
        round(rmse_y1_full, 4),
        round(rmse_y2_test, 4),
        round(rmse_y2_full, 4),
        round(rmse_lr_test, 4),
        round(rmse_ridge, 4),
        round(rmse_lasso, 4)
    ]
})

print("=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
print(summary.to_string(index=False))